# Final submission notebook — cleaned and memory-safe

This version moves imports to the top, keeps one shared data load, and makes Option C safer for WSL / Ubuntu shell runs.

In [ ]:
# Global imports, runtime config, and device setup
from pathlib import Path
import contextlib
import csv
import gc
import json
import math
import os
import sys
import time
import warnings

import cv2
import joblib
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image
from scipy.special import expit
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils import resample
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ----------------------------
# Runtime / memory profile
# ----------------------------
SAFE_MODE = True               # safer defaults for WSL / Ubuntu shell sessions
REBUILD_MEMMAP_CACHE = False   # set True only if the caches are stale/corrupted
RUN_OPTION_A = True
RUN_OPTION_B = True
RUN_OPTION_B_KAGGLE = True
RUN_OPTION_C = True
RUN_OPTION_C_KAGGLE = True

# Option C speed / stability toggles
OPTION_C_SCALING_MODE = "fast"     # "fast" = final split only, "full" = scaling curve
OPTION_C_ENABLE_TTA = False        # safer on RAM/VRAM; flip to True if stable
OPTION_C_USE_TORCH_COMPILE = False # torch.compile can spike memory and startup time
OPTION_C_BATCH_SIZE = 32 if SAFE_MODE else 64
OPTION_C_NUM_WORKERS = 0 if SAFE_MODE else 2
OPTION_C_PIN_MEMORY = torch.cuda.is_available()
OPTION_C_PREFETCH_FACTOR = None if OPTION_C_NUM_WORKERS == 0 else 2
OPTION_C_HEAD_EPOCHS = 8
OPTION_C_FULL_EPOCHS = 12

CUDA_DEVICE_INDEX = 0
_has_cuda = torch.cuda.is_available()
_has_mps = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())

if _has_cuda:
    torch.cuda.set_device(CUDA_DEVICE_INDEX)
    device_c = torch.device("cuda", CUDA_DEVICE_INDEX)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    try:
        _ = torch.zeros(2, 2, device=device_c, dtype=torch.float32).mean()
        torch.cuda.synchronize()
    except RuntimeError:
        device_c = torch.device("cpu")
elif _has_mps:
    device_c = torch.device("mps")
else:
    device_c = torch.device("cpu")

USE_AMP_C = device_c.type == "cuda"
print(f"torch {torch.__version__} | device_c={device_c}")
if device_c.type == "cuda":
    print(torch.cuda.get_device_name(CUDA_DEVICE_INDEX))

In [ ]:
# Paths and constants
REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / "data"
PART1_KAGGLE_DIR = DATA_ROOT / "part1" / "data" / "kaggle"
PART1_TRAIN_DIR = PART1_KAGGLE_DIR / "train" / "train"
PART1_TEST_DIR = PART1_KAGGLE_DIR / "test" / "test"

IMG_SIZE_CLASSICAL = (128, 128)
IMG_SIZE_CNN = (224, 224)
CLASS_NAMES = ["cat", "dog"]
LABEL_MAP = {"cat": 0, "dog": 1}

OUTPUTS_DIR = REPO_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
MODELS_DIR = OUTPUTS_DIR / "models"
CHECKPOINTS_DIR = OUTPUTS_DIR / "checkpoints"
SUBMISSIONS_DIR = OUTPUTS_DIR / "submissions"
TESTS_DIR = OUTPUTS_DIR / "tests"
CACHE_DIR = OUTPUTS_DIR / "cache"

for d in (FIGURES_DIR, MODELS_DIR, CHECKPOINTS_DIR, SUBMISSIONS_DIR, TESTS_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert PART1_TRAIN_DIR.exists(), f"Missing: {PART1_TRAIN_DIR}"
assert PART1_TEST_DIR.exists(), f"Missing: {PART1_TEST_DIR}"

print("Paths OK")
print(" train:", PART1_TRAIN_DIR)
print(" test: ", PART1_TEST_DIR)

## Shared helpers

In [ ]:
from src.utils import (
    extract_hog_features,
    resolve_svc,
    extract_multiscale_hog_hsv_features,
    clone_svc,
    load_test_images_memmap,
    load_labeled_images_memmap,
    load_labeled_images as src_load_labeled_images,
    get_pytorch_dataloaders as get_pytorch_dataloaders_memmap,
)

# ---------------------------------------------------------------------------
# Metrics (from src/evaluation.py)
# ---------------------------------------------------------------------------


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }


def compute_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    return confusion_matrix(y_true, y_pred)


def compare_models(results: dict) -> pd.DataFrame:
    df = pd.DataFrame(results).T
    df.index.name = "model"
    return df.sort_values("accuracy", ascending=False)


# ---------------------------------------------------------------------------
# Data loading (from src/utils.py)
# ---------------------------------------------------------------------------


def load_labeled_images(
    img_size: tuple,
    grayscale: bool,
    max_samples: int | None,
    return_ids: bool,
    return_paths: bool = False,
):
    # Build X in one preallocated array to limit peak RAM.
    path_order = []
    for class_name, label in LABEL_MAP.items():
        class_dir = PART1_TRAIN_DIR / f"{class_name}s"
        paths = sorted(class_dir.glob("*.jpg"))
        if max_samples is not None:
            paths = paths[:max_samples]
        for p in paths:
            path_order.append((p, label))

    h = img_size[0]
    w = img_size[1]
    n_paths = len(path_order)
    flag = cv2.IMREAD_GRAYSCALE if grayscale else cv2.IMREAD_COLOR

    if grayscale:
        X = np.empty((n_paths, h, w), dtype=np.float32)
    else:
        X = np.empty((n_paths, h, w, 3), dtype=np.float32)

    labels = np.empty(n_paths, dtype=np.int64)
    ids_list = []
    path_list = []
    write_i = 0

    for p, label in tqdm(path_order, desc="Loading train images"):
        img = cv2.imread(str(p), flag)
        if img is None:
            continue
        img = cv2.resize(img, (w, h))
        if not grayscale:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        X[write_i, ...] = img.astype(np.float32) / 255.0
        labels[write_i] = label
        ids_list.append(p.stem)
        path_list.append(p)
        write_i += 1

    if write_i < n_paths:
        X = X[:write_i].copy()
    y = labels[:write_i].copy()
    id_arr = np.array(ids_list, dtype=object)
    path_arr = np.array(path_list, dtype=object)
    if return_ids:
        if return_paths:
            return X, y, id_arr, path_arr
        return X, y, id_arr
    return X, y




def split_data(X, y, test_size: float, random_state: int):
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)


# PCA on torch GPU only if kernels work (arch mismatch: cuda available but ops fail)
_TORCH_PCA_USE_GPU = device_c.type in ("cuda", "mps")
if device_c.type == "cuda":
    try:
        _ = torch.zeros(2, 2, device=device_c, dtype=torch.float32).mean()
        torch.cuda.synchronize()
    except RuntimeError:
        _TORCH_PCA_USE_GPU = False


def downsample_rgb_gpu(X_nhwc: np.ndarray, size_hw: tuple) -> np.ndarray:
    if X_nhwc.size == 0:
        return X_nhwc
    arr = np.ascontiguousarray(X_nhwc, dtype=np.float32)
    dev = device_c if device_c.type in ("cuda", "mps") else torch.device("cpu")
    t = torch.from_numpy(arr).to(dev)
    t = t.permute(0, 3, 1, 2)
    if (t.shape[2], t.shape[3]) != (size_hw[0], size_hw[1]):
        try:
            t = torch.nn.functional.interpolate(
                t, size=(size_hw[0], size_hw[1]), mode="bilinear", align_corners=False
            )
        except RuntimeError:
            # PyTorch wheel missing kernels for this GPU (e.g. very new arch); use CPU
            t = torch.from_numpy(arr).to("cpu").permute(0, 3, 1, 2)
            t = torch.nn.functional.interpolate(
                t, size=(size_hw[0], size_hw[1]), mode="bilinear", align_corners=False
            )
    out = t.permute(0, 2, 3, 1).contiguous().cpu().numpy()
    return out.astype(np.float32)


def apply_pca_torch_gpu(X_train: np.ndarray, X_test: np.ndarray, n_components: int):
    dev = device_c if device_c.type in ("cuda", "mps") else torch.device("cpu")

    def _run(dev_inner):
        Xt = torch.from_numpy(np.ascontiguousarray(X_train, dtype=np.float32)).to(dev_inner)
        Xte = torch.from_numpy(np.ascontiguousarray(X_test, dtype=np.float32)).to(dev_inner)
        mean = Xt.mean(dim=0, keepdim=True)
        Xt_c = Xt - mean
        Xte_c = Xte - mean
        q = min(n_components, Xt_c.shape[0] - 1, Xt_c.shape[1])
        q = max(1, q)
        U, S, V = torch.pca_lowrank(Xt_c, q=q, center=False)
        Zt = Xt_c @ V
        Zte = Xte_c @ V
        return Zt.cpu().numpy().astype(np.float32), Zte.cpu().numpy().astype(np.float32)

    try:
        return _run(dev)
    except RuntimeError:
        # PyTorch wheel missing CUDA kernels for this GPU (e.g. Blackwell sm_120)
        if dev.type != "cpu":
            return _run(torch.device("cpu"))
        raise


def apply_pca_auto(X_train: np.ndarray, X_test: np.ndarray, n_components: int):
    if _TORCH_PCA_USE_GPU:
        try:
            return apply_pca_torch_gpu(X_train, X_test, n_components)
        except Exception:
            pass
    pca = PCA(n_components=n_components, random_state=42)
    X_tr = pca.fit_transform(X_train)
    X_te = pca.transform(X_test)
    return X_tr, X_te


def apply_pca(X_train, X_test, n_components: int):
    n_train = X_train.shape[0]
    n_test = X_test.shape[0]
    X_train_flat = X_train.reshape(n_train, -1)
    X_test_flat = X_test.reshape(n_test, -1)

    pca = PCA(n_components=n_components, random_state=42)
    X_train_pca = pca.fit_transform(X_train_flat)
    X_test_pca = pca.transform(X_test_flat)
    return X_train_pca, X_test_pca, pca


def standardize_features(X_train, X_test):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    return X_train_s, X_test_s, scaler


def generate_submission_csv(ids, predictions, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame({"id": ids, "label": predictions})
    df = df.sort_values("id").reset_index(drop=True)
    df.to_csv(output_path, index=False, lineterminator="\n")
    return output_path


# ---------------------------------------------------------------------------
# PyTorch dataloaders + GPU augmentation (from src/utils.py)
# ---------------------------------------------------------------------------


class _NumpyHWCDataset(Dataset):
    def __init__(self, X, y, img_size, do_normalize, mean3, std3):
        self.X = X
        self.y = y
        self.img_size = img_size
        self.do_normalize = do_normalize
        self._mean = mean3.view(3, 1, 1)
        self._std = std3.view(3, 1, 1)

    def __len__(self):
        return int(self.X.shape[0])

    def __getitem__(self, idx):
        x = self.X[idx]
        t = torch.from_numpy(np.ascontiguousarray(np.transpose(x, (2, 0, 1)))).float()
        if (t.shape[1], t.shape[2]) != self.img_size:
            t = torch.nn.functional.interpolate(
                t.unsqueeze(0),
                size=self.img_size,
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)
        if self.do_normalize:
            t.sub_(self._mean).div_(self._std)
        return t, int(self.y[idx])


def get_pytorch_dataloaders(
    X_train,
    y_train,
    X_val,
    y_val,
    batch_size: int,
    img_size: tuple,
    normalize_train: bool,
    pin_memory: bool = False,
    num_workers: int = 0,
):
    _m = torch.tensor([0.485, 0.456, 0.406])
    _s = torch.tensor([0.229, 0.224, 0.225])
    train_ds = _NumpyHWCDataset(X_train, y_train, img_size, normalize_train, _m, _s)
    val_ds = _NumpyHWCDataset(X_val, y_val, img_size, True, _m, _s)

    loader_kw = {"batch_size": batch_size, "pin_memory": pin_memory, "num_workers": num_workers}
    if num_workers > 0:
        loader_kw["persistent_workers"] = True
        loader_kw["prefetch_factor"] = 2

    train_loader = DataLoader(train_ds, shuffle=True, **loader_kw)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kw)
    return train_loader, val_loader


def build_gpu_augmentation(img_size: tuple):
    return transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            transforms.RandomErasing(p=0.1),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


# ---------------------------------------------------------------------------
# Plots (from src/visualization.py)
# ---------------------------------------------------------------------------


def plot_confusion_matrix_fig(
    cm: np.ndarray,
    class_names,
    title: str,
    save_path=None,
):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    fig.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
    plt.show()


def plot_sample_predictions_fig(
    images: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names,
    n: int,
    save_path=None,
):
    n = min(n, len(images))
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = axes.flatten()

    for i in range(n):
        ax = axes[i]
        img = images[i]
        if img.ndim == 2:
            ax.imshow(img, cmap="gray")
        else:
            ax.imshow(img)
        correct = y_true[i] == y_pred[i]
        color = "green" if correct else "red"
        ax.set_title(
            f"T:{class_names[y_true[i]]} P:{class_names[y_pred[i]]}",
            color=color,
            fontsize=9,
        )
        ax.axis("off")

    for j in range(n, len(axes)):
        axes[j].axis("off")

    fig.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
    plt.show()


def plot_model_comparison_fig(results: dict, save_path=None):
    df = pd.DataFrame(results).T
    ax = df.plot.bar(figsize=(10, 5), rot=25)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Internal validation — model configurations (Option B grid)")
    ax.legend(loc="lower right")
    fig = ax.get_figure()
    fig.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
    plt.show()


def plot_training_history_fig(history: dict, title=None, save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history["train_loss"]) + 1)

    ax1.plot(epochs, history["train_loss"], label="Train")
    ax1.plot(epochs, history["val_loss"], label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss")
    ax1.legend()

    ax2.plot(epochs, history["train_acc"], label="Train")
    ax2.plot(epochs, history["val_acc"], label="Val")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy")
    ax2.legend()

    if title is not None:
        fig.suptitle(title, fontsize=14)

    fig.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
    plt.show()


plot_training_history = plot_training_history_fig




PCA_VAR_Q_CAP = 2048


def _fit_pca_torch_train_test_b(X_train, X_test, n_components, dev):
    X_train = np.ascontiguousarray(X_train, dtype=np.float32)
    X_test = np.ascontiguousarray(X_test, dtype=np.float32)
    n_samples = X_train.shape[0]
    total_var = np.var(X_train, axis=0, ddof=1).sum()

    Xt = torch.from_numpy(X_train).to(dev)
    Xte = torch.from_numpy(X_test).to(dev)
    mean = Xt.mean(dim=0, keepdim=True)
    Xt_c = Xt - mean
    Xte_c = Xte - mean
    q = min(n_components, Xt_c.shape[0] - 1, Xt_c.shape[1])
    q = max(1, q)
    _U, S, V = torch.pca_lowrank(Xt_c, q=q, center=False)
    Zt = Xt_c @ V
    Zte = Xte_c @ V
    s_cpu = S.detach().cpu().numpy()
    ev = (s_cpu ** 2) / (n_samples - 1)
    evr = ev / total_var if total_var > 1e-12 else np.ones(q) / q

    class TorchPcaFacade:
        def __init__(self):
            self.mean_ = mean.cpu().numpy().astype(np.float64)
            self.components_ = V.T.cpu().numpy()
            self.explained_variance_ratio_ = evr.astype(np.float64)
            self._V = V.detach().cpu().numpy().astype(np.float32)
            self._mean_1d = self.mean_.reshape(-1)

        def transform(self, X):
            X = np.ascontiguousarray(X, dtype=np.float32)
            Xm = X - self._mean_1d
            return (Xm @ self._V).astype(np.float32)

    fac = TorchPcaFacade()
    return (
        Zt.detach().cpu().numpy().astype(np.float32),
        Zte.detach().cpu().numpy().astype(np.float32),
        fac,
    )


def _fit_pca_torch_train_only_b(X, n_components, dev):
    X = np.ascontiguousarray(X, dtype=np.float32)
    n_samples = X.shape[0]
    total_var = np.var(X, axis=0, ddof=1).sum()
    Xt = torch.from_numpy(X).to(dev)
    mean = Xt.mean(dim=0, keepdim=True)
    Xt_c = Xt - mean
    q = min(n_components, Xt_c.shape[0] - 1, Xt_c.shape[1])
    q = max(1, q)
    _U, S, V = torch.pca_lowrank(Xt_c, q=q, center=False)
    Z = Xt_c @ V
    s_cpu = S.detach().cpu().numpy()
    ev = (s_cpu ** 2) / (n_samples - 1)
    evr = ev / total_var if total_var > 1e-12 else np.ones(q) / q

    class TorchPcaFacadeFull:
        def __init__(self):
            self.mean_ = mean.cpu().numpy().astype(np.float64)
            self.components_ = V.T.cpu().numpy()
            self.explained_variance_ratio_ = evr.astype(np.float64)
            self._V = V.detach().cpu().numpy().astype(np.float32)
            self._mean_1d = self.mean_.reshape(-1)

        def transform(self, X):
            X = np.ascontiguousarray(X, dtype=np.float32)
            Xm = X - self._mean_1d
            return (Xm @ self._V).astype(np.float32)

    return Z.detach().cpu().numpy().astype(np.float32), TorchPcaFacadeFull()


def _fit_pca_torch_train_test_variance_b(
    X_train, X_test, variance_ratio, dev, q_cap=PCA_VAR_Q_CAP
):
    X_train = np.ascontiguousarray(X_train, dtype=np.float32)
    X_test = np.ascontiguousarray(X_test, dtype=np.float32)
    n_samples = X_train.shape[0]
    total_var = np.var(X_train, axis=0, ddof=1).sum()

    Xt = torch.from_numpy(X_train).to(dev)
    Xte = torch.from_numpy(X_test).to(dev)
    mean = Xt.mean(dim=0, keepdim=True)
    Xt_c = Xt - mean
    Xte_c = Xte - mean
    q = min(q_cap, Xt_c.shape[0] - 1, Xt_c.shape[1])
    q = max(1, q)
    _U, S, V = torch.pca_lowrank(Xt_c, q=q, center=False)
    s_cpu = S.detach().cpu().numpy()
    ev = (s_cpu ** 2) / (n_samples - 1)
    evr = ev / total_var if total_var > 1e-12 else np.ones(q) / q

    cum = np.cumsum(evr)
    k = len(cum)
    for j in range(len(cum)):
        if cum[j] >= variance_ratio:
            k = j + 1
            break

    V = V[:, :k]
    Zt = Xt_c @ V
    Zte = Xte_c @ V
    evr_k = evr[:k]

    class TorchPcaFacadeVar:
        def __init__(self):
            self.mean_ = mean.cpu().numpy().astype(np.float64)
            self.components_ = V.T.cpu().numpy()
            self.explained_variance_ratio_ = evr_k.astype(np.float64)
            self._V = V.detach().cpu().numpy().astype(np.float32)
            self._mean_1d = self.mean_.reshape(-1)

        def transform(self, X):
            X = np.ascontiguousarray(X, dtype=np.float32)
            Xm = X - self._mean_1d
            return (Xm @ self._V).astype(np.float32)

    fac = TorchPcaFacadeVar()
    return (
        Zt.detach().cpu().numpy().astype(np.float32),
        Zte.detach().cpu().numpy().astype(np.float32),
        fac,
    )


def _fit_pca_torch_train_only_variance_b(
    X, variance_ratio, dev, q_cap=PCA_VAR_Q_CAP
):
    X = np.ascontiguousarray(X, dtype=np.float32)
    n_samples = X.shape[0]
    total_var = np.var(X, axis=0, ddof=1).sum()
    Xt = torch.from_numpy(X).to(dev)
    mean = Xt.mean(dim=0, keepdim=True)
    Xt_c = Xt - mean
    q = min(q_cap, Xt_c.shape[0] - 1, Xt_c.shape[1])
    q = max(1, q)
    _U, S, V = torch.pca_lowrank(Xt_c, q=q, center=False)
    s_cpu = S.detach().cpu().numpy()
    ev = (s_cpu ** 2) / (n_samples - 1)
    evr = ev / total_var if total_var > 1e-12 else np.ones(q) / q

    cum = np.cumsum(evr)
    k = len(cum)
    for j in range(len(cum)):
        if cum[j] >= variance_ratio:
            k = j + 1
            break

    V = V[:, :k]
    Z = Xt_c @ V
    evr_k = evr[:k]

    class TorchPcaFacadeVarFull:
        def __init__(self):
            self.mean_ = mean.cpu().numpy().astype(np.float64)
            self.components_ = V.T.cpu().numpy()
            self.explained_variance_ratio_ = evr_k.astype(np.float64)
            self._V = V.detach().cpu().numpy().astype(np.float32)
            self._mean_1d = self.mean_.reshape(-1)

        def transform(self, X):
            X = np.ascontiguousarray(X, dtype=np.float32)
            Xm = X - self._mean_1d
            return (Xm @ self._V).astype(np.float32)

    return Z.detach().cpu().numpy().astype(np.float32), TorchPcaFacadeVarFull()


def apply_pca_features_auto(X_train, X_test, pca_spec):
    dev = device_c if device_c.type in ("cuda", "mps") else torch.device("cpu")
    if isinstance(pca_spec, tuple) and pca_spec[0] == "var":
        variance_ratio = pca_spec[1]
        if _TORCH_PCA_USE_GPU:
            try:
                return _fit_pca_torch_train_test_variance_b(
                    X_train, X_test, variance_ratio, dev
                )
            except Exception:
                pass
        pca = PCA(n_components=variance_ratio, random_state=42)
        X_tr = pca.fit_transform(X_train)
        X_te = pca.transform(X_test)
        return X_tr, X_te, pca

    n_components = int(pca_spec)
    if _TORCH_PCA_USE_GPU:
        try:
            return _fit_pca_torch_train_test_b(X_train, X_test, n_components, dev)
        except Exception:
            pass
    pca = PCA(n_components=n_components, random_state=42)
    X_tr = pca.fit_transform(X_train)
    X_te = pca.transform(X_test)
    return X_tr, X_te, pca


def fit_pca_train_only_auto(X, pca_spec):
    dev = device_c if device_c.type in ("cuda", "mps") else torch.device("cpu")
    if isinstance(pca_spec, tuple) and pca_spec[0] == "var":
        variance_ratio = pca_spec[1]
        if _TORCH_PCA_USE_GPU:
            try:
                return _fit_pca_torch_train_only_variance_b(X, variance_ratio, dev)
            except Exception:
                pass
        pca = PCA(n_components=variance_ratio, random_state=42)
        Z = pca.fit_transform(X)
        return Z, pca

    n_components = int(pca_spec)
    if _TORCH_PCA_USE_GPU:
        try:
            return _fit_pca_torch_train_only_b(X, n_components, dev)
        except Exception:
            pass
    pca = PCA(n_components=n_components, random_state=42)
    Z = pca.fit_transform(X)
    return Z, pca


def format_pca_spec_label(pca_spec):
    if isinstance(pca_spec, tuple) and pca_spec[0] == "var":
        return "var{:.2f}".format(pca_spec[1])
    return str(int(pca_spec))


print("Shared helpers loaded.")


## Part 1 — shared data

In [ ]:
# Part 1 — shared data load with aligned splits
PART1_RANDOM_STATE = 42
PART1_IMG_SIZE_A = (64, 64)
IMG_SIZE_OPT_B = (96, 96)
LETTERBOX_B = True

print("Loading train memmap at 224x224 once...")
X_all_224, y_all, train_ids_all = load_labeled_images_memmap(
    img_size=IMG_SIZE_CNN,
    grayscale=False,
    max_samples=None,
    return_ids=True,
    rebuild=REBUILD_MEMMAP_CACHE,
)

idx_all = np.arange(len(y_all), dtype=np.int64)
idx_train, idx_val = train_test_split(
    idx_all,
    test_size=0.2,
    random_state=PART1_RANDOM_STATE,
    stratify=y_all,
)

# Path lookup for qualitative plots
n_paths = len(train_ids_all)
PART1_train_paths = np.empty(n_paths, dtype=object)
for i, stem in enumerate(train_ids_all):
    stem = str(stem)
    subdir = "cats" if stem.startswith("cat") else "dogs"
    PART1_train_paths[i] = PART1_TRAIN_DIR / subdir / f"{stem}.jpg"

PART1_paths_val = PART1_train_paths[idx_val]

# Option A pixels are materialized once at low resolution
print("Preparing Option A 64x64 tensors from the aligned split...")
X_train_a_px = downsample_rgb_gpu(X_all_224[idx_train], PART1_IMG_SIZE_A)
X_test_a_px = downsample_rgb_gpu(X_all_224[idx_val], PART1_IMG_SIZE_A)
y_train_a = y_all[idx_train]
y_test_a = y_all[idx_val]

# Option C uses the original memmap + row indices directly
train_idx_c = idx_train.copy()
val_idx_c = idx_val.copy()
y_train_c = y_all[train_idx_c]
y_val_c = y_all[val_idx_c]

print("Loading Kaggle test at 224x224...")
X_test_224_c, kaggle_test_ids = load_test_images_memmap(
    img_size=IMG_SIZE_CNN,
    grayscale=False,
)

print(
    f"Full train memmap: {X_all_224.shape} | "
    f"train split: {len(idx_train)} | val split: {len(idx_val)} | "
    f"Option A pixels: {X_train_a_px.shape} | Kaggle: {X_test_224_c.shape}"
)

## Option A — HOG + SVM

In [ ]:
# Option A — parallel HOG + SVC (matches notebooks/01-feature-based-model-optionA.ipynb)

USE_GPU_A = True
CUDA_DEVICE_INDEX_A = 0
_cuda_ok_a = False
if USE_GPU_A and _has_cuda:
    torch.cuda.set_device(CUDA_DEVICE_INDEX_A)
    _dev_try = torch.device("cuda", CUDA_DEVICE_INDEX_A)
    try:
        _ = torch.zeros(2, 2, device=_dev_try, dtype=torch.float32).mean()
        torch.cuda.synchronize()
        _cw = torch.nn.Conv2d(3, 8, kernel_size=3, padding=1).to(_dev_try)
        _ = _cw(torch.zeros(1, 3, 16, 16, device=_dev_try))
        torch.cuda.synchronize()
        _cuda_ok_a = True
    except RuntimeError:
        _cuda_ok_a = False

SVC_A, SklearnSVC_A, _ = resolve_svc(_cuda_ok_a, "Option A", prefer_gpu=USE_GPU_A)
print(
    "Option A | USE_GPU_A={} | SVC: {}".format(
        USE_GPU_A,
        "cuml (GPU)" if SVC_A is not SklearnSVC_A else "sklearn (CPU)",
    )
)

FEAT_EXTRACT_N_JOBS_A = 1 if SAFE_MODE else -1
HOG_ORIENTATIONS_A = 9
HOG_PIXELS_PER_CELL_A = (8, 8)
HOG_CELLS_PER_BLOCK_A = (2, 2)

SVM_C_A = 10.0
SVM_KERNEL_A = "rbf"
SVM_GAMMA_A = "scale"
RANDOM_STATE_A = 42

print("Extracting HOG features (src.utils)...")
X_train_a = extract_hog_features(
    X_train_a_px,
    pixels_per_cell=HOG_PIXELS_PER_CELL_A,
    cells_per_block=HOG_CELLS_PER_BLOCK_A,
    orientations=HOG_ORIENTATIONS_A,
    n_jobs=FEAT_EXTRACT_N_JOBS_A,
)
X_test_a = extract_hog_features(
    X_test_a_px,
    pixels_per_cell=HOG_PIXELS_PER_CELL_A,
    cells_per_block=HOG_CELLS_PER_BLOCK_A,
    orientations=HOG_ORIENTATIONS_A,
    n_jobs=FEAT_EXTRACT_N_JOBS_A,
)
print(f"HOG dim: {X_train_a.shape[1]}")

_svm_kw_a = dict(
    C=SVM_C_A,
    kernel=SVM_KERNEL_A,
    gamma=SVM_GAMMA_A,
    class_weight="balanced",
    random_state=RANDOM_STATE_A,
)
if SVC_A is SklearnSVC_A:
    _svm_kw_a["decision_function_shape"] = "ovr"

model_a = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("svm", SVC_A(**_svm_kw_a)),
    ]
)

t0 = time.time()
model_a.fit(X_train_a, y_train_a)
train_time_min_a = (time.time() - t0) / 60.0
print(f"Option A training time: {train_time_min_a:.2f} min")

y_pred_a = model_a.predict(X_test_a)
metrics_a = compute_metrics(y_test_a, y_pred_a)
print("Option A metrics:", metrics_a)


def display_results_a(indices, title, num_samples=10):
    n = min(num_samples, len(indices))
    cols = 5
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(20, 5 * rows))
    for i in range(n):
        idx = indices[i]
        plt.subplot(rows, cols, i + 1)
        img_path = PART1_paths_val[idx]
        img = mpimg.imread(str(img_path))
        plt.imshow(img)
        is_correct = y_pred_a[idx] == y_test_a[idx]
        color = "green" if is_correct else "red"
        plt.title(
            f"True: {CLASS_NAMES[y_test_a[idx]]}\nPred: {CLASS_NAMES[y_pred_a[idx]]}",
            fontsize=12,
            color=color,
            fontweight="bold",
        )
        plt.axis("off")
    plt.suptitle(title, fontsize=22, y=1.02)
    plt.tight_layout()
    plt.show()


correct_idx_a = np.where(y_pred_a == y_test_a)[0]
incorrect_idx_a = np.where(y_pred_a != y_test_a)[0]
acc_a = metrics_a["accuracy"]
prec_a = metrics_a["precision"]
rec_a = metrics_a["recall"]
f1_a = metrics_a["f1"]
acc_text = f" (Accuracy: {acc_a * 100:.2f}%)"
display_results_a(correct_idx_a, f"Option A — correct{acc_text}")
display_results_a(incorrect_idx_a, f"Option A — errors{acc_text}")

cm_a = compute_confusion_matrix(y_test_a, y_pred_a)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_a,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    linewidths=0.5,
    ax=ax,
)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
ax.set_title("Option A — HOG + SVM confusion matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


results_a = {
    "Option A": {
        "accuracy": float(acc_a),
        "precision": float(prec_a),
        "recall": float(rec_a),
        "f1": float(f1_a),
        "train_time_min": float(train_time_min_a),
    }
}

print("Option A done.")


## Option B — appearance-based classical pipeline

In [ ]:
# Option B — matches notebooks/02-appearace-based-model-optionB.ipynb
extract_combined_features = extract_multiscale_hog_hsv_features

USE_GPU_B = True
SVC_B, SklearnSVC_B, _ = resolve_svc(device_c.type == "cuda", "Option B", prefer_gpu=USE_GPU_B)
print(
    "Option B: HOG+HSV+LBP + PCA + SVC | backend: {}".format(
        "cuml" if SVC_B is not SklearnSVC_B else "sklearn"
    )
)

IMG_SIZE_OPT_B = (96, 96)
LETTERBOX_B = True
USE_LABELED_MEMMAP_CACHE = True
MEMMAP_CACHE_DIR = OUTPUTS_DIR / "cache" / "part1_labeled_memmap"

if USE_LABELED_MEMMAP_CACHE:
    X_b_all, y_b_all, train_ids_b = load_labeled_images_memmap(
        img_size=IMG_SIZE_OPT_B,
        grayscale=False,
        return_ids=True,
        cache_dir=MEMMAP_CACHE_DIR,
        rebuild=REBUILD_MEMMAP_CACHE,
        letterbox=LETTERBOX_B,
    )
else:
    X_b_all, y_b_all, train_ids_b = src_load_labeled_images(
        img_size=IMG_SIZE_OPT_B,
        grayscale=False,
        return_ids=True,
        letterbox=LETTERBOX_B,
    )

X_train_b = X_b_all[idx_train]
X_test_b = X_b_all[idx_val]
y_train_b = y_b_all[idx_train]
y_test_b = y_b_all[idx_val]

FEAT_EXTRACT_N_JOBS_B = -1
HOG_ORIENTATIONS_B = 9
HOG_CELLS_PER_BLOCK_B = (2, 2)
HOG_SCALES_PPC_B = [(8, 8), (12, 12), (16, 16)]
HSV_BINS_B = 16
INCLUDE_LBP_B = True
LBP_N_POINTS_B = 24
LBP_RADIUS_B = 3

print("Train features (Option B)...")
X_train_feats_b = extract_combined_features(
    X_train_b,
    hog_orientations=HOG_ORIENTATIONS_B,
    hog_cells_per_block=HOG_CELLS_PER_BLOCK_B,
    hog_scales_ppc=HOG_SCALES_PPC_B,
    hsv_bins=HSV_BINS_B,
    n_jobs=FEAT_EXTRACT_N_JOBS_B,
    include_lbp=INCLUDE_LBP_B,
    lbp_n_points=LBP_N_POINTS_B,
    lbp_radius=LBP_RADIUS_B,
)
print("Test features (Option B)...")
X_test_feats_b = extract_combined_features(
    X_test_b,
    hog_orientations=HOG_ORIENTATIONS_B,
    hog_cells_per_block=HOG_CELLS_PER_BLOCK_B,
    hog_scales_ppc=HOG_SCALES_PPC_B,
    hsv_bins=HSV_BINS_B,
    n_jobs=FEAT_EXTRACT_N_JOBS_B,
    include_lbp=INCLUDE_LBP_B,
    lbp_n_points=LBP_N_POINTS_B,
    lbp_radius=LBP_RADIUS_B,
)
print(f"Feature dim: {X_train_feats_b.shape[1]}")

X_train_s_b, X_test_s_b, feat_scaler_b = standardize_features(X_train_feats_b, X_test_feats_b)

pca_specs_b = []
for n in [200, 300, 400, 500]:
    pca_specs_b.append(n)
for vr in [0.95, 0.99]:
    pca_specs_b.append(("var", vr))

classifier_specs_b = [
    (
        "SVC-RBF",
        SVC_B(
            kernel="rbf",
            C=10.0,
            gamma="scale",
            class_weight="balanced",
            random_state=42,
        ),
    ),
]

results_b = {}
best_accuracy_b = -1.0
best_pca_spec_b = None
best_pca_b = None
best_clf_b = None
best_clf_label_b = None
best_y_pred_b = None

t0_b_train = time.time()
for pca_spec in pca_specs_b:
    X_tr_pca, X_te_pca, pca = apply_pca_features_auto(
        X_train_s_b,
        X_test_s_b,
        pca_spec,
    )

    for clf_label, clf_template in classifier_specs_b:
        clf = clone_svc(clf_template, SklearnSVC_B)
        clf.fit(X_tr_pca, y_train_b)
        y_pred = clf.predict(X_te_pca)
        metrics = compute_metrics(y_test_b, y_pred)
        pca_tag = format_pca_spec_label(pca_spec)
        name = "HOG+HSV+LBP+PCA-{}+{}".format(pca_tag, clf_label)
        results_b[name] = metrics
        print("{} -> {}".format(name, metrics))

        acc = metrics["accuracy"]
        if acc > best_accuracy_b:
            best_accuracy_b = acc
            best_pca_spec_b = pca_spec
            best_pca_b = pca
            best_clf_b = clf
            best_clf_label_b = clf_label
            best_y_pred_b = y_pred

print("")
print(
    "Best Option B: PCA spec={!r}, clf={}, accuracy={:.4f}".format(
        best_pca_spec_b,
        best_clf_label_b,
        best_accuracy_b,
    )
)

train_time_min_b = (time.time() - t0_b_train) / 60.0
print(f"Option B grid + sweep time: {train_time_min_b:.2f} min")

metrics_b = compute_metrics(y_test_b, best_y_pred_b)
comparison_df_b = compare_models(results_b)
comparison_df_b


results_b["Option A"] = results_a["Option A"]
print("Option B done.")


In [ ]:
# Option B — plots
plot_model_comparison_fig(results_b, save_path=FIGURES_DIR / "optionB_grid_metrics.png")

best_cm_b = compute_confusion_matrix(y_test_b, best_y_pred_b)
_pca_lbl = format_pca_spec_label(best_pca_spec_b)
plot_confusion_matrix_fig(
    best_cm_b,
    CLASS_NAMES,
    title=(
        f"Option B best: HOG+HSV+LBP+PCA-{_pca_lbl} + {best_clf_label_b}"
    ),
    save_path=FIGURES_DIR / "confusion_matrix_optionB.png",
)

plot_sample_predictions_fig(
    X_test_b,
    y_test_b,
    best_y_pred_b,
    CLASS_NAMES,
    n=16,
    save_path=FIGURES_DIR / "sample_predictions_optionB.png",
)

cumulative_ev_b = np.cumsum(best_pca_b.explained_variance_ratio_)
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cumulative_ev_b) + 1), cumulative_ev_b)
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance")
_pca_lbl = format_pca_spec_label(best_pca_spec_b)
plt.title(f"PCA (Option B), spec={_pca_lbl}, clf={best_clf_label_b}")
plt.grid(True)
plt.savefig(FIGURES_DIR / "pca_variance_optionB.png", dpi=150)
plt.show()
print(f"Variance explained (last component): {cumulative_ev_b[-1]:.3f}")


In [ ]:
# Option B — retrain on all data, write Kaggle CSVs (optional hflip TTA)
USE_TEST_TIME_FLIP_B = True

print("Full-train features (Option B)...")
X_all_feats_b = extract_combined_features(
    X_b_all,
    hog_orientations=HOG_ORIENTATIONS_B,
    hog_cells_per_block=HOG_CELLS_PER_BLOCK_B,
    hog_scales_ppc=HOG_SCALES_PPC_B,
    hsv_bins=HSV_BINS_B,
    n_jobs=FEAT_EXTRACT_N_JOBS_B,
    include_lbp=INCLUDE_LBP_B,
    lbp_n_points=LBP_N_POINTS_B,
    lbp_radius=LBP_RADIUS_B,
)

X_all_s_b, _, scaler_final_b = standardize_features(X_all_feats_b, X_all_feats_b)

X_all_pca_b, pca_final_b = fit_pca_train_only_auto(X_all_s_b, best_pca_spec_b)

clf_final_b = clone_svc(best_clf_b, SklearnSVC_B)
clf_final_b.fit(X_all_pca_b, y_b_all)
print("Option B final model fit on all labels.")

print("Loading Kaggle test (letterboxed 96×96)...")
X_kaggle_test_b, kaggle_ids_b = load_test_images_memmap(
    img_size=IMG_SIZE_OPT_B,
    grayscale=False,
    letterbox=LETTERBOX_B,
)

print("Kaggle features (Option B)...")
X_kaggle_feats_b = extract_combined_features(
    X_kaggle_test_b,
    hog_orientations=HOG_ORIENTATIONS_B,
    hog_cells_per_block=HOG_CELLS_PER_BLOCK_B,
    hog_scales_ppc=HOG_SCALES_PPC_B,
    hsv_bins=HSV_BINS_B,
    n_jobs=FEAT_EXTRACT_N_JOBS_B,
    include_lbp=INCLUDE_LBP_B,
    lbp_n_points=LBP_N_POINTS_B,
    lbp_radius=LBP_RADIUS_B,
)

X_kaggle_s_b = scaler_final_b.transform(X_kaggle_feats_b)
X_kaggle_pca_b = pca_final_b.transform(X_kaggle_s_b)


def _dog_proba_from_clf_b(clf, Xp):
    if hasattr(clf, "predict_proba") and getattr(clf, "probability", True):
        return clf.predict_proba(Xp)[:, 1]
    return expit(clf.decision_function(Xp))


kaggle_pred_proba_b = _dog_proba_from_clf_b(clf_final_b, X_kaggle_pca_b)

if USE_TEST_TIME_FLIP_B:
    X_kaggle_flip_b = np.flip(X_kaggle_test_b, axis=2)
    X_kaggle_feats_f_b = extract_combined_features(
        X_kaggle_flip_b,
        hog_orientations=HOG_ORIENTATIONS_B,
        hog_cells_per_block=HOG_CELLS_PER_BLOCK_B,
        hog_scales_ppc=HOG_SCALES_PPC_B,
        hsv_bins=HSV_BINS_B,
        n_jobs=FEAT_EXTRACT_N_JOBS_B,
        include_lbp=INCLUDE_LBP_B,
        lbp_n_points=LBP_N_POINTS_B,
        lbp_radius=LBP_RADIUS_B,
    )
    X_kaggle_s_f_b = scaler_final_b.transform(X_kaggle_feats_f_b)
    X_kaggle_pca_f_b = pca_final_b.transform(X_kaggle_s_f_b)
    p_f_b = _dog_proba_from_clf_b(clf_final_b, X_kaggle_pca_f_b)
    kaggle_pred_proba_b = (kaggle_pred_proba_b + p_f_b) / 2.0

kaggle_pred_b = (kaggle_pred_proba_b >= 0.5).astype(np.int64)

submission_b = generate_submission_csv(
    kaggle_ids_b,
    kaggle_pred_b,
    SUBMISSIONS_DIR / "optionB_multihog_hsv_pca_rf_submission.csv",
)
debug_b = generate_submission_csv(
    kaggle_ids_b,
    kaggle_pred_proba_b,
    TESTS_DIR / "optionB_multihog_hsv_pca_rf_submission_proba.csv",
)
print("Submission:", submission_b)
print("Debug proba:", debug_b)


# Free some heavy classical arrays before Option C
for _name in [
    "X_train_feats_b", "X_test_feats_b", "X_all_feats_b", "X_kaggle_feats_b",
    "X_kaggle_feats_f_b", "X_kaggle_s_b", "X_kaggle_pca_b", "X_all_pca_b",
    "X_all_s_b", "X_b_all", "X_train_b", "X_test_b", "X_kaggle_test_b"
]:
    if _name in globals():
        del globals()[_name]
gc.collect()
if device_c.type == "cuda":
    torch.cuda.empty_cache()
print("Freed Option B intermediates.")


## Option C — deep learning (memory-safe rewrite)

In [ ]:
# Option C — memory-safe datasets, augmentation, and train/eval helpers

class _NumpyTestHWCDataset(Dataset):
    def __init__(self, X, img_size, normalize=True):
        self.X = X
        self.img_size = img_size
        self.normalize = normalize
        self._mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self._std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return int(self.X.shape[0])

    def __getitem__(self, idx):
        x = self.X[idx]
        t = torch.from_numpy(np.ascontiguousarray(np.transpose(x, (2, 0, 1)))).float()
        if (t.shape[1], t.shape[2]) != self.img_size:
            t = torch.nn.functional.interpolate(
                t.unsqueeze(0), size=self.img_size, mode="bilinear", align_corners=False
            ).squeeze(0)
        if self.normalize:
            t.sub_(self._mean).div_(self._std)
        return t


def make_test_loader(X, batch_size, img_size, num_workers=0, pin_memory=False):
    ds = _NumpyTestHWCDataset(X, img_size=img_size, normalize=True)
    loader_kw = dict(batch_size=batch_size, shuffle=False, pin_memory=pin_memory, num_workers=num_workers)
    if num_workers > 0:
        loader_kw["persistent_workers"] = False
        loader_kw["prefetch_factor"] = OPTION_C_PREFETCH_FACTOR
    return DataLoader(ds, **loader_kw)


def maybe_compile_c(model):
    if not OPTION_C_USE_TORCH_COMPILE or not hasattr(torch, "compile"):
        return model
    try:
        return torch.compile(model, mode="default")
    except Exception as exc:
        print("torch.compile skipped:", exc)
        return model


def build_model_c(dev):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, 2))
    model = model.to(dev)
    if dev.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
    return maybe_compile_c(model)


# Lighter than the previous notebook: avoids RandomResizedCrop + RandomErasing spikes
gpu_augment_c = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def train_one_epoch_c(model, loader, criterion, optimizer, dev, augment=None, scaler=None, use_amp=False):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    non_blocking = OPTION_C_PIN_MEMORY and dev.type == "cuda"
    amp_ctx = (
        torch.amp.autocast("cuda", enabled=True)
        if use_amp and dev.type == "cuda"
        else contextlib.nullcontext()
    )

    for images, labels in tqdm(loader, desc="  Train", leave=False):
        # Apply light augmentation before device transfer to reduce VRAM spikes
        if augment is not None:
            images = torch.stack([augment(img) for img in images], dim=0)
        images = images.to(dev, non_blocking=non_blocking)
        labels = labels.to(dev, non_blocking=non_blocking)
        if dev.type == "cuda":
            images = images.contiguous(memory_format=torch.channels_last)

        optimizer.zero_grad(set_to_none=True)
        with amp_ctx:
            outputs = model(images)
            loss = criterion(outputs, labels)

        if scaler is not None and use_amp and dev.type == "cuda":
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += float(loss.item()) * images.size(0)
        correct += int((outputs.argmax(dim=1) == labels).sum().item())
        total += int(labels.size(0))

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate_c(model, loader, criterion, dev, use_amp=False):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    non_blocking = OPTION_C_PIN_MEMORY and dev.type == "cuda"
    amp_ctx = (
        torch.amp.autocast("cuda", enabled=True)
        if use_amp and dev.type == "cuda"
        else contextlib.nullcontext()
    )

    for images, labels in tqdm(loader, desc="  Val  ", leave=False):
        images = images.to(dev, non_blocking=non_blocking)
        labels = labels.to(dev, non_blocking=non_blocking)
        if dev.type == "cuda":
            images = images.contiguous(memory_format=torch.channels_last)
        with amp_ctx:
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += float(loss.item()) * images.size(0)
        correct += int((outputs.argmax(dim=1) == labels).sum().item())
        total += int(labels.size(0))

    return running_loss / total, correct / total


@torch.no_grad()
def predict_loader_c(model, loader, dev, use_amp=False, flip_tta=False):
    model.eval()
    all_probs = []
    non_blocking = OPTION_C_PIN_MEMORY and dev.type == "cuda"
    amp_ctx = (
        torch.amp.autocast("cuda", enabled=True)
        if use_amp and dev.type == "cuda"
        else contextlib.nullcontext()
    )

    for batch in tqdm(loader, desc="Predict", leave=False):
        images = batch[0] if isinstance(batch, (tuple, list)) else batch
        images = images.to(dev, non_blocking=non_blocking)
        if dev.type == "cuda":
            images = images.contiguous(memory_format=torch.channels_last)

        with amp_ctx:
            logits = model(images)
            probs = torch.softmax(logits, dim=1)

        if flip_tta:
            flipped = torch.flip(images, dims=[3])
            with amp_ctx:
                logits_f = model(flipped)
                probs_f = torch.softmax(logits_f, dim=1)
            probs = 0.5 * (probs + probs_f)

        all_probs.append(probs.cpu())

    return torch.cat(all_probs, dim=0).numpy()

In [ ]:
# Option C — progressive training on the aligned Part 1 split without duplicating the 224 memmap

if OPTION_C_SCALING_MODE == "full":
    SAMPLE_SIZES_C = [500, 1000, 2000, 4000, len(train_idx_c)]
else:
    SAMPLE_SIZES_C = [len(train_idx_c)]

print(f"Option C sample sizes: {SAMPLE_SIZES_C}")
print(
    f"Option C loader config | batch={OPTION_C_BATCH_SIZE} | workers={OPTION_C_NUM_WORKERS} | "
    f"pin_memory={OPTION_C_PIN_MEMORY} | amp={USE_AMP_C}"
)

grad_scaler_c = torch.amp.GradScaler("cuda", enabled=(USE_AMP_C and device_c.type == "cuda"))

loader_kw_c = dict(
    batch_size=OPTION_C_BATCH_SIZE,
    img_size=IMG_SIZE_CNN,
    normalize_train=False,
    lazy_from_numpy=True,
    num_workers=OPTION_C_NUM_WORKERS,
    pin_memory=OPTION_C_PIN_MEMORY,
    persistent_workers=False,
    prefetch_factor=OPTION_C_PREFETCH_FACTOR,
)

def run_training_phases_c(model, train_loader, val_loader, criterion, grad_scaler, use_amp):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_state = None

    head_params = [p for p in model.fc.parameters() if p.requires_grad]
    optimizer = optim.Adam(head_params, lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=OPTION_C_HEAD_EPOCHS)

    print(f"  Phase 1: head only for {OPTION_C_HEAD_EPOCHS} epochs")
    for epoch in range(1, OPTION_C_HEAD_EPOCHS + 1):
        tl, ta = train_one_epoch_c(
            model, train_loader, criterion, optimizer, device_c,
            augment=gpu_augment_c, scaler=grad_scaler, use_amp=use_amp,
        )
        vl, va = evaluate_c(model, val_loader, criterion, device_c, use_amp=use_amp)
        history["train_loss"].append(tl)
        history["train_acc"].append(ta)
        history["val_loss"].append(vl)
        history["val_acc"].append(va)
        scheduler.step()
        print(f"    Epoch {epoch}/{OPTION_C_HEAD_EPOCHS} — TrL:{tl:.4f} TrA:{ta:.4f} VL:{vl:.4f} VA:{va:.4f}")
        if va > best_val_acc:
            best_val_acc = va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    for param in model.parameters():
        param.requires_grad = True

    optimizer_ft = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
    scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=OPTION_C_FULL_EPOCHS)

    print(f"  Phase 2: full fine-tuning for {OPTION_C_FULL_EPOCHS} epochs")
    for epoch in range(1, OPTION_C_FULL_EPOCHS + 1):
        tl, ta = train_one_epoch_c(
            model, train_loader, criterion, optimizer_ft, device_c,
            augment=gpu_augment_c, scaler=grad_scaler, use_amp=use_amp,
        )
        vl, va = evaluate_c(model, val_loader, criterion, device_c, use_amp=use_amp)
        history["train_loss"].append(tl)
        history["train_acc"].append(ta)
        history["val_loss"].append(vl)
        history["val_acc"].append(va)
        scheduler_ft.step()
        print(f"    Epoch {epoch}/{OPTION_C_FULL_EPOCHS} — TrL:{tl:.4f} TrA:{ta:.4f} VL:{vl:.4f} VA:{va:.4f}")
        if va > best_val_acc:
            best_val_acc = va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return best_state, best_val_acc, history


cv_progressive_results_c = []
best_state_oof_c = None
t0_c_train = time.time()

for n_samples in SAMPLE_SIZES_C:
    print(f"\n--- Progressive size {n_samples} / {len(train_idx_c)} ---")
    if n_samples < len(train_idx_c):
        pool = np.arange(len(train_idx_c), dtype=np.int64)
        sub_pool = resample(
            pool,
            n_samples=n_samples,
            stratify=y_train_c,
            random_state=42,
        )
        idx_train_sub = train_idx_c[sub_pool]
        y_train_sub = y_all[idx_train_sub]
    else:
        idx_train_sub = train_idx_c
        y_train_sub = y_train_c

    train_loader_c, val_loader_c = get_pytorch_dataloaders_memmap(
        X_all_224,
        y_train_sub,
        X_all_224,
        y_val_c,
        train_row_indices=idx_train_sub,
        val_row_indices=val_idx_c,
        **loader_kw_c,
    )

    model_c = build_model_c(device_c)
    criterion_c = nn.CrossEntropyLoss(label_smoothing=0.05)
    best_state, best_val_acc, history = run_training_phases_c(
        model_c, train_loader_c, val_loader_c, criterion_c, grad_scaler_c, USE_AMP_C
    )

    best_state_oof_c = best_state
    cv_progressive_results_c.append(
        {"n_train": int(n_samples), "best_val_acc": float(best_val_acc), "history": history}
    )
    print(f"  ✓ Best val accuracy: {best_val_acc:.4f}")

    del train_loader_c, val_loader_c, model_c, criterion_c
    gc.collect()
    if device_c.type == "cuda":
        torch.cuda.empty_cache()

assert best_state_oof_c is not None, "Option C training did not produce a checkpoint."

model_c = build_model_c(device_c)
model_c.load_state_dict(best_state_oof_c)
model_c.eval()

val_loader_eval_c = get_pytorch_dataloaders_memmap(
    X_all_224,
    y_train_c[:1],
    X_all_224,
    y_val_c,
    train_row_indices=train_idx_c[:1],
    val_row_indices=val_idx_c,
    **loader_kw_c,
)[1]

val_probs_c = predict_loader_c(model_c, val_loader_eval_c, device_c, use_amp=USE_AMP_C, flip_tta=False)
val_pred_c = val_probs_c.argmax(axis=1)

metrics_c = compute_metrics(y_val_c, val_pred_c)
cm_c = compute_confusion_matrix(y_val_c, val_pred_c)
final_history_c = cv_progressive_results_c[-1]["history"]
train_time_min_c = (time.time() - t0_c_train) / 60.0

results_c = {
    "Option C": {
        **metrics_c,
        "train_time_min": float(train_time_min_c),
        "best_val_acc": float(cv_progressive_results_c[-1]["best_val_acc"]),
    }
}

print("Option C metrics:", json.dumps(results_c["Option C"], indent=2))

In [ ]:
# Option C — plots and save
plot_training_history(
    final_history_c,
    title="Option C — training history",
    save_path=FIGURES_DIR / "optionC_training_history.png",
)

plot_confusion_matrix_fig(
    cm_c,
    CLASS_NAMES,
    title="Option C — confusion matrix",
    save_path=FIGURES_DIR / "confusion_matrix_optionC.png",
)

# Use the original 224 images for qualitative samples
plot_sample_predictions_fig(
    X_all_224[val_idx_c],
    y_val_c,
    val_pred_c,
    CLASS_NAMES,
    n=16,
    save_path=FIGURES_DIR / "sample_predictions_optionC.png",
)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
option_c_model_path = MODELS_DIR / "option_c_resnet50_memory_safe.pt"
torch.save(best_state_oof_c, option_c_model_path)

with open(MODELS_DIR / "option_c_metrics.json", "w") as f:
    json.dump(results_c["Option C"], f, indent=2)

print("Saved:", option_c_model_path)
print("Saved:", MODELS_DIR / "option_c_metrics.json")

In [ ]:
# Option C — Kaggle submission without materializing the full test set as one giant torch tensor
if RUN_OPTION_C_KAGGLE:
    test_loader_c = make_test_loader(
        X_test_224_c,
        batch_size=OPTION_C_BATCH_SIZE,
        img_size=IMG_SIZE_CNN,
        num_workers=OPTION_C_NUM_WORKERS,
        pin_memory=OPTION_C_PIN_MEMORY,
    )
    kaggle_probs_c = predict_loader_c(
        model_c,
        test_loader_c,
        device_c,
        use_amp=USE_AMP_C,
        flip_tta=OPTION_C_ENABLE_TTA,
    )
    kaggle_pred_c = kaggle_probs_c.argmax(axis=1)

    submission_option_c = generate_submission_csv(
        kaggle_test_ids,
        kaggle_pred_c,
        SUBMISSIONS_DIR / "submission_optionC_memory_safe.csv",
    )
    print("Saved:", submission_option_c)
    print(f" Dogs: {(kaggle_pred_c == 1).sum()} | Cats: {(kaggle_pred_c == 0).sum()}")

## Final summary

In [ ]:
# Final comparison table
all_results = {}
if "results_a" in globals():
    all_results.update(results_a)
if "results_b" in globals():
    all_results.update(results_b)
if "results_c" in globals():
    all_results.update(results_c)

comparison_df = compare_models(all_results)
display(comparison_df)

comparison_df.to_csv(OUTPUTS_DIR / "model_comparison_summary.csv")
print("Saved:", OUTPUTS_DIR / "model_comparison_summary.csv")

In [ ]:
from pathlib import Path
import sys

# Add project root so src imports work when running from notebooks/.
sys.path.append(str(Path().resolve().parent))

from src.config import (
    PART1_KAGGLE_DIR,
    PART2_IMAGES_DIR,
    PART2_ANNOTATIONS_DIR,
    validate_data_layout,
    MODELS_DIR,
    OPTION_C_CNN,
    LOCALIZATION_DIR
)

validate_data_layout()

print("Data paths validated:")
print(f"- Part 1: {PART1_KAGGLE_DIR}")
print(f"- Part 2 Images: {PART2_IMAGES_DIR}")
print(f"- Part 2 Annotations: {PART2_ANNOTATIONS_DIR}")
print(f"- Model Directories: {MODELS_DIR}")

### Setup

In [ ]:
import numpy as np
import torch
import cv2
import os
import gc
import xml.etree.ElementTree as ET
from tqdm import tqdm
from torchvision import transforms
from torchvision import models
from PIL import Image

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plot_training_history
import matplotlib.pyplot as plt

from src.config import (
    IMG_SIZE_CNN,
    CLASS_NAMES,
    FIGURES_DIR,
    MODELS_DIR,
    LOCALIZATION_DIR,
    OUTPUTS_DIR,
)

from src.utils import (
    load_labeled_images,
    split_data,
    get_pytorch_dataloaders,
    load_test_images,
    generate_submission_csv,
    build_gpu_augmentation,
)

from src.evaluation import compute_metrics, compute_confusion_matrix
from src.visualization import (
    plot_confusion_matrix,
    plot_training_history,
    plot_sample_predictions,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(f"Using device: {device}")

### Defining our Grad-CAM Model

In [ ]:
class GradCAM_Locator:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        # Register the hooks to grab data during the forward and backward passes
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_ouput):
        self.gradients = grad_ouput[0].detach()
    
    #Heat map and bounding box together
    def get_heatmap_and_bbox(self, image_tensor, original_image_shape, threshold=0.0):
        self.model.eval()
        self.gradients = None
        self.activations = None

        # Forward
        output = self.model(image_tensor)
        dog_class_index = 1
        score = output[:, dog_class_index]

        # Backward
        self.model.zero_grad(set_to_none=True)
        score.backward()

        if self.gradients is None or self.activations is None:
            return None, None

        # Grad-CAM
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activations = self.activations.clone()
        activations *= pooled_gradients.view(1, -1, 1, 1)

        #Heatmap generation
        heatmap = torch.mean(activations, dim=1).squeeze()
        heatmap = F.relu(heatmap)

        max_val = torch.max(heatmap)
        if max_val > 0:
            heatmap = heatmap / max_val

        heatmap = heatmap.detach().cpu().numpy()

        # Resize to original image size
        heatmap_resized = cv2.resize(heatmap, (original_image_shape[1], original_image_shape[0]))

        # Bounding box from heatmap
        binary_map = (heatmap_resized > threshold).astype(np.uint8)

        ys, xs = np.where(binary_map > 0)
        if len(xs) == 0 or len(ys) == 0:
            return heatmap_resized, None

        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()

        bbox = [int(x_min), int(y_min), int(x_max - x_min), int(y_max - y_min)]

        return heatmap_resized, bbox

### Loading our model

In [ ]:
model_path = MODELS_DIR / OPTION_C_CNN
# print(model_path)
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = True
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 2),
)
model.load_state_dict(torch.load(model_path, weights_only=True, map_location=device))
model.to(device)
model.eval()

### Loading all Image paths

In [ ]:
all_image_paths = []

for breed_folder in os.listdir(PART2_IMAGES_DIR):
    breed_path = PART2_IMAGES_DIR / breed_folder

    if os.path.isdir(breed_path):
        for img_name in os.listdir(breed_path):
            if img_name.endswith(".jpg"):
                all_image_paths.append(breed_path / img_name)

print(f"Found {len(all_image_paths)} images")

### Analysis helper functions

In [ ]:
def compute_iou(boxA, boxB):
    # determine the box = [x, y, w, h]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
	
	# compute the area of intersection rectangle
    inter_area = max(0, xB - xA) * max(0, yB - yA)

    areaA = boxA[2] * boxA[3]
    areaB = boxB[2] * boxB[3]

	# compute union
    union = areaA + areaB - inter_area

    if union == 0:
        return 0.0
    
	# compute the intersection over union
    return inter_area / union

In [ ]:
def load_gt_box(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    boxes = []
    # Extract coordinates of each box
    for bbox in root.findall(".//bndbox"):
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)
        boxes.append((xmin, ymin, xmax, ymax))

    if not boxes:
        return None

    # Merge all boxes into one
    xmin = min(b[0] for b in boxes)
    ymin = min(b[1] for b in boxes)
    xmax = max(b[2] for b in boxes)
    ymax = max(b[3] for b in boxes)

    return [xmin, ymin, xmax - xmin, ymax - ymin]

In [ ]:
# Given an image path, return the corresponding annotation path
def get_annotation_path(image_path):
    image_path = Path(image_path)
    #Breed
    breeds = image_path.parent.name
    #Filename without extension
    stem = image_path.stem
    return PART2_ANNOTATIONS_DIR / breeds / stem

In [ ]:
# Attach the hooks to the target conv layer
target_layer = model.layer4[2].conv3
cam_extractor = GradCAM_Locator(model, target_layer)

### Quantitative Evaluation

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE_CNN)),
    transforms.ToTensor(),
])

In [ ]:
ious = []

for idx, image_path in enumerate(tqdm(all_image_paths)):
    # Load image
    img = cv2.imread(str(image_path))
    if img is None:
        continue
    
    #Original shape for resizing heatmap later
    original_shape = img.shape[:2]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess and prepare tensor for Grad-CAM
    img_tensor = transform(img_rgb).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)

    # Get corresponding annotation path
    xml_path = get_annotation_path(image_path)

    # Load ground truth box from XML
    gt_box = load_gt_box(xml_path)

    # Generate heatmap and predicted box using Grad-CAM
    with torch.enable_grad():
        _, pred_box = cam_extractor.get_heatmap_and_bbox(img_tensor, original_shape, threshold=0)

    # Compute IoU and store
    if pred_box is None:
        ious.append(0.0)
    else:
        iou = compute_iou(pred_box, gt_box)
        ious.append(iou)

    # Free memory
    del img, img_rgb, img_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print("IoUs collected:", len(ious))

In [ ]:
ious = np.array(ious)

# Compute metrics
mean_iou = np.mean(ious)
acc_poor = np.mean(ious < 0.5) * 100
acc_50 = np.mean((ious >= 0.5) & (ious < 0.7)) * 100
acc_70 = np.mean((ious >= 0.7) & (ious < 0.9)) * 100
acc_90 = np.mean(ious >= 0.9) * 100

# Outputs
print(f"Mean IoU: {mean_iou:.4f}")
print(f"Frequency of Poor IoU: {acc_poor:.2f}%")
print(f"Frequency of Average IoU: {acc_50:.2f}%")
print(f"Frequency of Good IoU: {acc_70:.2f}%")
print(f"Frequency of Excellent IoU: {acc_90:.2f}%")

### Qualitative Evaluation of Best and Worse performing dogs

In [ ]:
worst_k = 50
best_k = 50
worst_cases = []
best_cases = []

for image_path in tqdm(all_image_paths):
    # Load image
    img = cv2.imread(str(image_path))
    if img is None:
        continue
    
    #Original shape for resizing heatmap later
    original_shape = img.shape[:2]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess and prepare tensor for Grad-CAM
    img_tensor = transform(img_rgb).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)

    # Get corresponding annotation path
    xml_path = get_annotation_path(image_path)

    # Load ground truth box from XML
    gt_box = load_gt_box(xml_path)

    # Extract predicted bounding box
    _,pred_box = cam_extractor.get_heatmap_and_bbox(img_tensor, original_shape, threshold=0)

    # Compute IoU and store
    if pred_box is None:
        iou = 0.0
        pred_box_to_store = None
    else:
        iou = compute_iou(pred_box, gt_box)
        pred_box_to_store = pred_box.copy()   # important

    # Store case details for worst and best cases
    case = {
        "iou": float(iou),
        "image_path": str(image_path),
        "pred_box": pred_box_to_store,
        "gt_box": gt_box.copy()
    }

    # Update worst and best cases
    worst_cases.append(case)
    worst_cases = sorted(worst_cases, key=lambda x: x["iou"])[:worst_k]

    best_cases.append(case)
    best_cases = sorted(best_cases, key=lambda x: x["iou"], reverse=True)[:best_k]

    del img, img_rgb, img_tensor

### Print 50 worse performing cases

In [ ]:
print("=== Worst 50 cases ===")
breed_count = {}
for i, case in enumerate(worst_cases, 1):
    print(f"{i}. IoU={case['iou']:.4f}")
    print(f"   image: {case['image_path']}")
    breed = str(Path(case['image_path']).parent).split('-')[-1]
    print(f"    breed: {breed}")
    if breed not in breed_count:
        breed_count.update({breed: 1})
    else:
        breed_count.update({breed: breed_count[breed]+1})

# Sort by the value (x[1]) in descending order
sorted_breed_count = dict(sorted(breed_count.items(), key=lambda x: x[1], reverse=True))

print(sorted_breed_count)

### Display Worse Bounding Boxes

In [ ]:
plt.figure(figsize=(128, 128))
print(len(worst_cases))
for i, case in enumerate(worst_cases[:50]):
    # Load image
    img = cv2.imread(case["image_path"])
    vis = img.copy()

    # Get boxes
    pred_box = case["pred_box"]
    gt_box = case["gt_box"]

    # Draw predicted
    if pred_box is not None:
        x, y, w, h = pred_box
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Draw GT
    gx, gy, gw, gh = gt_box
    cv2.rectangle(vis, (gx, gy), (gx + gw, gy + gh), (0, 0, 255), 2)

    # Display
    plt.subplot(10, 5, i + 1)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"IoU={case['iou']:.2f}")
    plt.axis('off')

plt.suptitle("50 Worst IoU Cases (Predicted = Green, GT = Red)", fontsize=16)
plt.tight_layout()
plt.show()

### Display corresponding Heat Maps

In [ ]:
plt.figure(figsize=(128, 128))

for i, case in enumerate(worst_cases[:50]):
    # Load image
    img = cv2.imread(case["image_path"])
    if img is None:
        continue
    
    #Original shape for resizing heatmap later
    original_shape = img.shape[:2]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess and prepare tensor for Grad-CAM
    img_tensor = transform(img_rgb).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)

    # Get predicted bounding box and heatmap
    heatmap, pred_box = cam_extractor.get_heatmap_and_bbox(img_tensor, original_shape, threshold=0)

    # overlay heatmap
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap),cv2.COLORMAP_JET)

    # visible image on Display
    vis = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)

    # predicted (green)
    if pred_box is not None:
        x, y, w, h = pred_box
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # GT (red)
    gx, gy, gw, gh = case["gt_box"]
    cv2.rectangle(vis, (gx, gy), (gx + gw, gy + gh), (0, 0, 255), 2)

    # Display
    plt.subplot(10, 5, i + 1)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"IoU={case['iou']:.2f}")
    plt.axis('off')

plt.suptitle("Worst 50 Cases with Grad-CAM Heatmaps", fontsize=16)
plt.tight_layout()
plt.show()

### Qualitative Evaluation of Best performing Dogs

In [ ]:
print("=== Best 50 cases ===")
breed_count = {}

for i, case in enumerate(best_cases, 1):
    print(f"{i}. IoU={case['iou']:.4f}")
    print(f"   image: {case['image_path']}")
    breed = str(Path(case['image_path']).parent).split('-')[-1]
    print(f"    breed: {breed}")
    if breed not in breed_count:
        breed_count.update({breed: 1})
    else:
        breed_count.update({breed: breed_count[breed]+1})

# Sort by the value (x[1]) in descending order
sorted_breed_count = dict(sorted(breed_count.items(), key=lambda x: x[1], reverse=True))

print(sorted_breed_count)

In [ ]:
plt.figure(figsize=(128, 128))

for i, case in enumerate(best_cases[:50]):
    # Load image
    img = cv2.imread(case["image_path"])
    vis = img.copy()

    # Get boxes
    pred_box = case["pred_box"]
    gt_box = case["gt_box"]

    #Draw predicted
    if pred_box is not None:
        x, y, w, h = pred_box
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Draw GT
    gx, gy, gw, gh = gt_box
    cv2.rectangle(vis, (gx, gy), (gx + gw, gy + gh), (0, 0, 255), 2)

    # Display
    plt.subplot(10, 5, i + 1)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"IoU={case['iou']:.2f}")
    plt.axis('off')

plt.suptitle("50 Best IoU Cases (Predicted = Green, GT = Red)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(128, 128))

for i, case in enumerate(best_cases[:50]):
    # Load image
    img = cv2.imread(case["image_path"])
    if img is None:
        continue
    
    #Original shape for resizing heatmap later
    original_shape = img.shape[:2]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess and prepare tensor for Grad-CAM
    img_tensor = transform(img_rgb).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)

    # Get predicted bounding box and heatmap
    heatmap, pred_box = cam_extractor.get_heatmap_and_bbox(img_tensor, original_shape, threshold=0)

    # overlay heatmap
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap),cv2.COLORMAP_JET)

    # visible image on Display
    vis = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)

    # predicted (green)
    if pred_box is not None:
        x, y, w, h = pred_box
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # GT (red)
    gx, gy, gw, gh = case["gt_box"]
    cv2.rectangle(vis, (gx, gy), (gx + gw, gy + gh), (0, 0, 255), 2)

    # Display
    plt.subplot(10, 5, i + 1)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"IoU={case['iou']:.2f}")
    plt.axis('off')

plt.suptitle("Best 10 Cases with Grad-CAM Heatmaps", fontsize=16)
plt.tight_layout()
plt.show()